In [16]:
from google.colab import files
files.upload()  # এখানে "Choose Files" দিয়ে kaggle.json select করো


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"marufhasan009","key":"79116af4bf72e85b4b45a941ee8738c8"}'}

In [17]:
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [18]:
!kaggle datasets download -d omkargurav/face-mask-dataset -p /content/sample_data/Mask_detection


Dataset URL: https://www.kaggle.com/datasets/omkargurav/face-mask-dataset
License(s): unknown
 75% 123M/163M [00:00<00:00, 1.29GB/s]
100% 163M/163M [00:00<00:00, 1.20GB/s]


In [19]:
import zipfile

with zipfile.ZipFile("/content/sample_data/Mask_detection/face-mask-dataset.zip", "r") as zip_ref:
    zip_ref.extractall("/content/sample_data/Mask_detection/")


In [20]:
import os
import shutil
from sklearn.model_selection import train_test_split

In [21]:
# source folder
data_dir = '/content/sample_data/Mask_detection/data'
classes = ['with_mask','without_mask']

In [22]:
#Create folder for train/test
base_dir= '/content/sample_data/Mask_dataset'
train_dir = os.path.join(base_dir,"train")
test_dir = os.path.join(base_dir,"test")

In [23]:
# create folder structure
for folder in [train_dir,test_dir]:
  for cls in classes:
    os.makedirs(os.path.join(folder,cls),exist_ok=True)

In [24]:
# split data
for cls in classes:
  cls_path = os.path.join(data_dir,cls)
  images = os.listdir(cls_path)

  #80%  train, 20% test

  train_img,test_img = train_test_split(images, test_size= 0.2,random_state=42)

  #copy train images
  for img in train_img:
    src= os.path.join(cls_path,img)
    m_src= os.path.join(train_dir,cls,img)
    shutil.copyfile(src,m_src)

  #copy test image
  for img in test_img:
    src = os.path.join(cls_path,img)
    m_src = os.path.join(test_dir,cls,img)
    shutil.copyfile(src,m_src)

print('train/test split done')


train/test split done


In [25]:
# Genarator
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen= ImageDataGenerator(
    rescale =1./255,
    rotation_range = 20,
    width_shift_range = 0.1,
    horizontal_flip = True

)

test_datagen = ImageDataGenerator(rescale=1./255)

train_genarator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(64,64),
    batch_size =32,
    class_mode='binary'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(64,64),
    batch_size=32,
    class_mode='binary'
)

Found 6042 images belonging to 2 classes.
Found 1511 images belonging to 2 classes.


In [26]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow import keras
from keras import Sequential

In [27]:
model = Sequential([
    Conv2D(32,(3,3),activation='relu',input_shape=(64,64,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128,activation='relu'),
    Dropout(0.5),

    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
model.compile(
    optimizer= 'adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [29]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       589,952 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 683,329 (2.61 MB)

 Trainable params: 683,329 (2.61 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

In [31]:
checkpoint = ModelCheckpoint(
    filepath='best_mask_model.h5',
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [32]:
callbacks = [early_stop,checkpoint]

In [34]:
history = model.fit(
    train_genarator,
    validation_data=test_generator,
    epochs=25,
    callbacks=callbacks
)

Epoch 1/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9656 - loss: 0.0941
Epoch 1: val_loss improved from 0.11432 to 0.11271, saving model to best_mask_model.h5


189/189 ━━━━━━━━━━━━━━━━━━━━ 19s 99ms/step - accuracy: 0.9656 - loss: 0.0941 - val_accuracy: 0.9676 - val_loss: 0.1127
Epoch 2/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9669 - loss: 0.1015
Epoch 2: val_loss did not improve from 0.11271
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9669 - loss: 0.1015 - val_accuracy: 0.9471 - val_loss: 0.1502
Epoch 3/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9658 - loss: 0.0972
Epoch 3: val_loss did not improve from 0.11271
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9658 - loss: 0.0972 - val_accuracy: 0.9636 - val_loss: 0.1173
Epoch 4/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9686 - loss: 0.0903
Epoch 4: val_loss improved from 0.11271 to 0.10959, saving model to best_mask_model.h5


189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 95ms/step - accuracy: 0.9686 - loss: 0.0903 - val_accuracy: 0.9616 - val_loss: 0.1096
Epoch 5/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9718 - loss: 0.0827
Epoch 5: val_loss did not improve from 0.10959
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 93ms/step - accuracy: 0.9717 - loss: 0.0828 - val_accuracy: 0.9649 - val_loss: 0.1202
Epoch 6/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.9715 - loss: 0.0880
Epoch 6: val_loss improved from 0.10959 to 0.10112, saving model to best_mask_model.h5


189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 96ms/step - accuracy: 0.9715 - loss: 0.0880 - val_accuracy: 0.9742 - val_loss: 0.1011
Epoch 7/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9761 - loss: 0.0723
Epoch 7: val_loss did not improve from 0.10112
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 92ms/step - accuracy: 0.9761 - loss: 0.0723 - val_accuracy: 0.9590 - val_loss: 0.1780
Epoch 8/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.9703 - loss: 0.0814
Epoch 8: val_loss did not improve from 0.10112
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - accuracy: 0.9703 - loss: 0.0814 - val_accuracy: 0.9616 - val_loss: 0.1560
Epoch 9/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9790 - loss: 0.0567
Epoch 9: val_loss improved from 0.10112 to 0.09702, saving model to best_mask_model.h5


189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 94ms/step - accuracy: 0.9789 - loss: 0.0567 - val_accuracy: 0.9669 - val_loss: 0.0970
Epoch 10/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.9775 - loss: 0.0656
Epoch 10: val_loss did not improve from 0.09702
189/189 ━━━━━━━━━━━━━━━━━━━━ 18s 96ms/step - accuracy: 0.9775 - loss: 0.0656 - val_accuracy: 0.9629 - val_loss: 0.1421
Epoch 11/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step - accuracy: 0.9768 - loss: 0.0739
Epoch 11: val_loss did not improve from 0.09702
189/189 ━━━━━━━━━━━━━━━━━━━━ 17s 91ms/step - accuracy: 0.9768 - loss: 0.0739 - val_accuracy: 0.9722 - val_loss: 0.1101
Epoch 12/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.9776 - loss: 0.0593
Epoch 12: val_loss did not improve from 0.09702
189/189 ━━━━━━━━━━━━━━━━━━━━ 20s 108ms/step - accuracy: 0.9776 - loss: 0.0593 - val_accuracy: 0.9735 - val_loss: 0.1138
Epoch 13/25
189/189 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.9770 - loss: 0.0596
Epoch 13: val_loss did not 

In [36]:
loss, acc = model.evaluate(test_generator)

48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 119ms/step - accuracy: 0.9683 - loss: 0.1040


In [37]:
model.save("final_mask_model.h5")

In [40]:
!pip install huggingface_hub
from huggingface_hub import login

login()


In [41]:
from huggingface_hub import HfApi, HfFolder, upload_file

repo_name = "Maruf-mask-detector"
api = HfApi()

api.create_repo(repo_name, private=False)

RepoUrl('https://huggingface.co/Maruf998/Maruf-mask-detector', endpoint='https://huggingface.co', repo_type='model', repo_id='Maruf998/Maruf-mask-detector')

In [43]:
upload_file(
    path_or_fileobj="final_mask_model.h5",
    path_in_repo="final_mask_model.h5",
    repo_id=f"Maruf998/{repo_name}",
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  final_mask_model.h5         :   7%|6         |  575kB / 8.25MB            

CommitInfo(commit_url='https://huggingface.co/Maruf998/Maruf-mask-detector/commit/17e80b51c2b803cd40396214fed8d75f6f380a8c', commit_message='Upload final_mask_model.h5 with huggingface_hub', commit_description='', oid='17e80b51c2b803cd40396214fed8d75f6f380a8c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Maruf998/Maruf-mask-detector', endpoint='https://huggingface.co', repo_type='model', repo_id='Maruf998/Maruf-mask-detector'), pr_revision=None, pr_num=None)